In [ ]:
! pip3 install matplotlib numpy pandas pylzma ipykernel jupyter torch

In [ ]:
! pip3 install ipykernel jupyter torch

In [ ]:
! python3 -m ipykernel install --user --name=.venv --display-name="venv-gpt"

In [ ]:
! jupyter notebook

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'mps' if torch.mps.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250
# dropout = 0.2

In [ ]:
with open('wizard_of_oz.txt', 'r', encoding = 'utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

In [ ]:
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

In [ ]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    x,y = x.to(device), y.to(device)

    return x,y

x,y = get_batch('train')
print('inputs: ')
print(x.shape)
print(x)
print('targets:')
print(y)

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is', context, 'target is', target)

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embeddings_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        logits = self.token_embeddings_table(index)

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) 

        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B,T) array of indices in the current context

        for _ in range(max_new_tokens):
            # Get the predictions
            logits, loss = self.forward(index)
            # Focus only on the last time step
            logits = logits[:, -1, :] # (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim = -1) # (B,C)
            # sample form the distribution
            index_next = torch.multinomial(probs, num_samples= 1) # (B,1)
            #append the sampled index to the running sequence
            index = torch.cat((index, index_next), dim = 1) # (B,T+1)
        return index
        

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
genererated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(genererated_chars)

In [ ]:
#Create a Pytorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step {iter}, train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}")
    
    # Sample a batch of data
    xb, yb = get_batch('train')


    # evaluate the loss
    logits, loss = model.forward(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

In [ ]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
genererated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(genererated_chars)